# 05 — Designing CPs by Composing Operators

The interesting structure of an SRG is determined entirely by the *input* tiling. By pre-composing Conway operators on top of a regular tiling you can quickly explore a wide design space.

This notebook is a **recipe book**: each cell shows one composition, with a one-line description.

In [ ]:
import os
os.environ.setdefault('TQDM_DISABLE', '1')

import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    classifiers,
    colorization,
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    shrink_rotate,
    rendering,
)
from eucare.rendering import multi_show


In [ ]:
from eucare.search_trees import face_bfs_tree
from eucare.shrink_rotate import assign_this_way_by_face_z_order, shrink_rotate_pattern


def srg_pipeline(G):
    """Run the standard SRG pipeline: BFS z-order -> SRG -> recompute."""
    central = min(G.faces, key=lambda f: np.linalg.norm(f.midpoint()))
    central['z_order'] = 0
    for orig, dest in face_bfs_tree(central):
        dest['z_order'] = orig['z_order'] + 1
    assign_this_way_by_face_z_order(G)
    SRG = shrink_rotate_pattern(G)
    return SRG


In [ ]:
def show_pair(G, title):
    """Render a tiling and its SRG side-by-side."""
    SRG = srg_pipeline(G)
    multi_show([G, SRG],
               titles=[f'tiling: {title}', 'SRG'],
               face_inset=0.04, render_vertices=False)


## Visualizing operator fundamental domains

Every `GeometricConwayOperator` is built from a small fundamental-domain graph with three corner vertices `(v1, vf, v2)`. `op.show()` renders that domain, colouring elements by role:

- **orange** — the three triangle corners,
- **red** — vertices/edges marked `delete=True` (removed during substitution),
- **green** — vertices marked `join=True` (collapsed if order-2 after substitution),
- **grey** — retained elements.

Pass `annotate_barycentric=True` to also print each vertex's barycentric coordinates relative to `(v1, vf, v2)`.

In [ ]:
names = []
operators = []
for name in dir(conway):
    if name.endswith('_graph'):
        operators.append(getattr(conway, name)())
        title = name[:-6].replace('_', ' ')
        names.append(title)

# sort by number of edges which are not deleted in the fundamental domain, then by name
operators, names = zip(*sorted(
    zip(operators, names), 
    key=lambda pair: (len([h for h in pair[0].graph.halfedges if not h.attributes.get('delete', False)]), pair[1])
))

multi_show(
    operators,
    titles=names,
    ncols=3,
)

## Recipe 1: ambo on hexagons

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(6), rings=2)
G = conway.ambo_graph()(G, delete_on_border=False)
G.recompute_lengths_and_angles()
show_pair(G, 'ambo(hex)')


## Recipe 2: kis on the 3.3.4.3.4 Archimedean tiling

In [ ]:
G = example_graphs.from_tiles(example_tilesets.t_3_3_4_3_4(), rings=2)
G = conway.kis_graph()(G, delete_on_border=True)
G.recompute_lengths_and_angles()
show_pair(G, 'kis(3.3.4.3.4)')


## Recipe 3: alternating-flagstone on a square tiling

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=2)
G = conway.alternating_flagstone_graph()(G, delete_on_border=True)
G.recompute_lengths_and_angles()
show_pair(G, 'alt-flagstone(squares)')


## Recipe 4: Pietro-Vitelli flagstones on hexagons

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(6), rings=2)
G = conway.flagstone_pvitelli_graph()(G, delete_on_border=True)
G.recompute_lengths_and_angles()
show_pair(G, 'flagstone_pvitelli(hex)')


## Going further

- Conway operators compose freely, experiment with applying one after the other.
- Experiment with different archimedean tilings in `example_tilesets` as starting points; or with those from `curved_platonic` (see `02_Curved_Geometries`). 
- Many algorithms to construct crease patterns for tesselations from a tiling can be implemented as a two step process: First, apply a certain operators which adds all the neccessary creases (the correct topology of the crease pattern), secondly move the vertices in the crease patterns to their correct positions. This process is used for shrink-rotate tesselations (TODO:reference notbook and conway operator), intersecting cylinder type tesselations (TODO: reference notebook and conway operator) as well as alternating flagstones (TODO: reference notebook and conway operator). Which other kinds of origami tesselations may be possible to construct this way?